# 07. Instrumental Variables

## Background

Propensity scores and DiD require either no unmeasured confounding or a before/after structure. Instrumental variables offer a third route: find a variable Z that (1) is correlated with treatment T, (2) only affects outcome Y through T (exclusion restriction), and (3) is independent of all confounders. Such a variable "instruments" variation in T that is uncontaminated by confounding.

The canonical examples are famous: proximity to a college instruments for education (Card 1995); draft lottery numbers instrument for Vietnam service (Angrist 1990); quarter of birth instruments for schooling (Angrist & Krueger 1991). Each exploits a source of variation in treatment that is arguably exogenous — not driven by the same confounders as the treatment itself.

## What You'll Learn

- IV assumptions: relevance (first stage F > 10), exclusion restriction, independence
- Two-Stage Least Squares (2SLS): the standard IV estimator
- Local Average Treatment Effect (LATE): IV identifies effects only for compliers
- Weak instruments: F-statistic diagnostic and consequences
- Heterogeneous effects and the monotonicity assumption

## Why This Matters

2SLS is implemented in virtually every major statistics package (statsmodels `IV2SLS`, R `ivreg`). The exclusion restriction is untestable in exactly-identified models — it must be defended on subject-matter grounds. Weak instruments (F < 10) inflate variance and bias the 2SLS estimate toward OLS — the Stock & Yogo (2005) rule of thumb for the first-stage F is now standard practice. Reporting weak-instrument-robust confidence intervals (Anderson-Rubin) is increasingly required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.sandbox.regression.gmm import IV2SLS
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. IV Assumptions and the LATE Framework

With a binary instrument Z and binary treatment T, there are four "compliance types":
- **Compliers**: T(1)=1, T(0)=0 — take treatment iff encouraged
- **Always-takers**: T(1)=T(0)=1
- **Never-takers**: T(1)=T(0)=0
- **Defiers**: T(1)=0, T(0)=1 (ruled out by monotonicity)

IV identifies the ATE only for compliers: LATE = ITT_Y / ITT_T

In [ ]:
def simulate_iv(n=2000, late=4.0, instrument_strength=0.4, seed=0):
    """
    IV DGP with unobserved confounder U.
    Z = binary instrument (randomly assigned encouragement)
    T = treatment (endogenous — driven by U and Z)
    Y = outcome (driven by T, U, and noise)
    """
    rng = np.random.default_rng(seed)

    # Unobserved confounder
    U = rng.normal(0, 1, n)
    # Instrument: randomly assigned (e.g., randomized encouragement)
    Z = rng.binomial(1, 0.5, n)

    # First stage: T depends on Z and U
    logit_t = -1 + instrument_strength * 4 * Z + 0.8 * U
    p_t = 1 / (1 + np.exp(-logit_t))
    T = rng.binomial(1, p_t)

    # Outcome: depends on T, U (confounded), but not Z directly (exclusion)
    Y = 2 + late * T + 1.5 * U + rng.normal(0, 1, n)

    return pd.DataFrame({'Z': Z, 'T': T, 'Y': Y, 'U': U})


df_iv = simulate_iv(n=3000, late=4.0, instrument_strength=0.4)

# Naïve OLS (biased)
ols = smf.ols('Y ~ T', data=df_iv).fit()
ols_est = ols.params['T']

# Manual Wald estimator (binary Z, binary T)
def wald_estimator(df):
    """ITT_Y / ITT_T — simple Wald IV for binary instrument."""
    itt_y = df.loc[df.Z==1,'Y'].mean() - df.loc[df.Z==0,'Y'].mean()
    itt_t = df.loc[df.Z==1,'T'].mean() - df.loc[df.Z==0,'T'].mean()
    return itt_y / itt_t, itt_y, itt_t

late_wald, itt_y, itt_t = wald_estimator(df_iv)
print(f"True LATE          = 4.000")
print(f"Naïve OLS          = {ols_est:.3f}  (biased upward by confounder U)")
print(f"Wald IV estimate   = {late_wald:.3f}")
print(f"  ITT_Y = {itt_y:.3f}, ITT_T = {itt_t:.3f}")
compliance_rate = itt_t
print(f"  Compliance rate ≈ {compliance_rate:.1%}")

## 2. Two-Stage Least Squares (2SLS)

With continuous instruments or multiple instruments, Wald doesn't generalize. 2SLS does:

**Stage 1**: Regress T on Z (and controls). Get fitted values T̂.
**Stage 2**: Regress Y on T̂ (and controls).

The T̂ purges the endogenous variation in T, leaving only the instrument-driven variation.

In [ ]:
def two_stage_ls(Y, T, Z, W=None):
    """
    Manual 2SLS implementation.
    Y: outcome, T: endogenous regressor, Z: instrument, W: exogenous controls.
    """
    n = len(Y)
    if W is None:
        # Stage 1
        X1 = np.column_stack([np.ones(n), Z])
        stage1_coef = np.linalg.lstsq(X1, T, rcond=None)[0]
        T_hat = X1 @ stage1_coef
        f_stat = (np.var(T_hat) / np.var(T - T_hat)) * (n - 2) / 1
    else:
        X1 = np.column_stack([np.ones(n), Z, W])
        stage1_coef = np.linalg.lstsq(X1, T, rcond=None)[0]
        T_hat = X1 @ stage1_coef
        # F-stat for instrument (partial F)
        X1_noz = np.column_stack([np.ones(n), W])
        T_hat_noz = X1_noz @ np.linalg.lstsq(X1_noz, T, rcond=None)[0]
        sse_r = np.sum((T - T_hat_noz)**2)
        sse_u = np.sum((T - T_hat)**2)
        k_extra = Z.shape[1] if Z.ndim > 1 else 1
        f_stat = ((sse_r - sse_u) / k_extra) / (sse_u / (n - X1.shape[1]))

    # Stage 2
    X2 = np.column_stack([np.ones(n), T_hat])
    stage2_coef = np.linalg.lstsq(X2, Y, rcond=None)[0]
    T_hat_full = np.column_stack([np.ones(n), T]) @ stage2_coef
    # Correct SE using actual T, not T_hat (sandwich)
    resid = Y - np.column_stack([np.ones(n), T]) @ stage2_coef
    s2 = np.sum(resid**2) / (n - 2)
    XtX_inv = np.linalg.inv(X2.T @ X2)
    se = np.sqrt(s2 * XtX_inv[1, 1])

    return dict(tau=stage2_coef[1], se=se, f_stat=f_stat)


result_2sls = two_stage_ls(df_iv.Y.values, df_iv.T.values, df_iv.Z.values)
ci = (result_2sls['tau'] - 1.96*result_2sls['se'],
      result_2sls['tau'] + 1.96*result_2sls['se'])
print(f"2SLS estimate: {result_2sls['tau']:.3f}  (SE={result_2sls['se']:.3f})")
print(f"95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"First-stage F = {result_2sls['f_stat']:.1f}  (threshold: F > 10)")

## 3. Weak Instruments

When the instrument has little correlation with treatment (low first-stage F), 2SLS becomes unreliable — its finite-sample distribution has fat tails and mean that doesn't exist. The Stock & Yogo (2005) threshold of F > 10 is a standard diagnostic.

In [ ]:
# Simulate across a range of instrument strengths
instrument_strengths = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]
n_sims = 200
true_late = 4.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

f_stats_all = []
estimates_all = []

for strength in instrument_strengths:
    f_s, ests = [], []
    for sim in range(n_sims):
        df_s = simulate_iv(n=500, late=true_late, instrument_strength=strength, seed=sim*13)
        r = two_stage_ls(df_s.Y.values, df_s.T.values, df_s.Z.values)
        f_s.append(r['f_stat'])
        ests.append(r['tau'])
    f_stats_all.append(np.median(f_s))
    estimates_all.append(ests)

# Plot 1: F-statistic vs instrument strength
axes[0].plot(instrument_strengths, f_stats_all, 'o-', color='steelblue', lw=2)
axes[0].axhline(10, color='red', ls='--', lw=2, label='F=10 threshold')
axes[0].set_xlabel('Instrument strength'); axes[0].set_ylabel('Median first-stage F')
axes[0].set_title('Instrument Strength → F-statistic')
axes[0].legend()

# Plot 2: distribution of estimates by strength
positions = range(len(instrument_strengths))
bp = axes[1].boxplot(estimates_all, positions=list(positions), patch_artist=True,
                     medianprops=dict(color='black', lw=2))
for patch, strength in zip(bp['boxes'], instrument_strengths):
    patch.set_facecolor('#F44336' if f_stats_all[instrument_strengths.index(strength)] < 10 else '#4CAF50')
    patch.set_alpha(0.7)
axes[1].axhline(true_late, color='black', lw=2, ls='--', label=f'True LATE={true_late}')
axes[1].set_xticks(list(positions))
axes[1].set_xticklabels([f'{s}' for s in instrument_strengths])
axes[1].set_xlabel('Instrument strength'); axes[1].set_ylabel('2SLS estimate')
axes[1].set_title('2SLS Estimates — Red=weak instrument (F<10)')
axes[1].legend()

plt.tight_layout()
plt.savefig('07_weak_instruments.png', dpi=110, bbox_inches='tight')
plt.show()
print("Red boxes: weak instrument (F<10) — estimates unstable and biased toward OLS")

## 4. IV Assumptions Checklist

In [ ]:
print("=== IV Assumptions Checklist ===")
print()
print("1. RELEVANCE: Cov(Z, T) ≠ 0")
print("   Test: First-stage F-statistic > 10  (Stock & Yogo 2005)")
print("   Consequence if violated: weak instrument — 2SLS blows up")
print()
print("2. EXCLUSION RESTRICTION: Z → Y only through T")
print("   Test: UNTESTABLE in just-identified models")
print("   Requires: subject-matter reasoning + domain expertise")
print("   Consequence if violated: 2SLS is biased (no statistical fix)")
print()
print("3. INDEPENDENCE: Z ⊥ (U, Y(0), Y(1))")
print("   Test: check balance of Z across observed covariates")
print("   Consequence if violated: Z is confounded → 2SLS biased")
print()
print("4. MONOTONICITY: T_i(Z=1) >= T_i(Z=0) for all i  [no defiers]")
print("   Test: untestable, but partial: one-sided first stage helps")
print("   Consequence if violated: LATE interpretation breaks down")
print()
print("5. SUTVA: stable potential outcomes for instrument too")

# Illustrate exclusion restriction violation
print("\n--- Exclusion restriction violation ---")
rng = np.random.default_rng(99)
n = 2000
U = rng.normal(0, 1, n)
Z = rng.binomial(1, 0.5, n)
T = rng.binomial(1, 1/(1+np.exp(-(-1 + 1.5*Z + 0.8*U))), n)
# Z directly affects Y (violation!)
Y_viol = 2 + 3*T + 1.5*U + 1.5*Z + rng.normal(0, 1, n)   # Z→Y directly!
res_viol = two_stage_ls(Y_viol, T, Z)
print(f"True LATE=3.0, but Z directly adds 1.5 to Y (exclusion violated)")
print(f"2SLS estimate = {res_viol['tau']:.3f}  (biased!)")